# Issue Pattern & Root Cause Explorer

**Goal:** What are the most common complaint themes and how are they shifting?

- **Data:** CFPB complaints — raw data lands in `landing/cfpb_complaints/` (Parquet), then dlt loads into DuckDB and dbt builds marts.
- **Use this notebook** to explore the issue-focused marts and to get query patterns for your **BI Desktop** presentation layer (Treemap, Heatmap, small multiples, state map, filters).

## 1. Data source and dbt models

| Mart | Use in BI |
|------|-----------|
| `marts.fact_complaints` | Detail for filters (Product, State, Time) |
| `marts.issue_product_summary` | **Treemap** (issue frequency), **Heatmap** (Issue × Product) |
| `marts.issue_state_summary` | **State map**, cross-tab Issue × State |
| `marts.issue_growth_monthly` | **Small multiples** time-series by issue |
| `marts.issue_dispute_ratio` | % disputed per issue (root cause / theme) |

In [5]:
import os
import duckdb
import pandas as pd

DB_PATH = "database/cfpb_complaints.duckdb"
if not os.path.exists(DB_PATH):
    DB_PATH = os.path.join(os.path.dirname(os.getcwd()), "database", "cfpb_complaints.duckdb")
if not os.path.exists(DB_PATH):
    raise FileNotFoundError("Run dbt and ensure database/cfpb_complaints.duckdb exists (or run from project root).")

conn = duckdb.connect(DB_PATH, read_only=True)

## 2. Treemap — Issue frequency

Use **`issue_product_summary`** or **`issue_dispute_ratio`** (issue-level totals). For Treemap by issue only, aggregate by `issue` or use `issue_dispute_ratio`.

In [6]:
# Treemap: issue frequency (one row per issue, size = complaint_count)
# Uses dim_issues issue-level rollups (sub_issue IS NULL). After dbt run, marts.issue_dispute_ratio also works.
treemap_issue = conn.execute("""
    SELECT issue, total_complaints AS complaint_count
    FROM marts.dim_issues
    WHERE sub_issue IS NULL
    ORDER BY total_complaints DESC
""").df()
treemap_issue.head(15)

,issue,complaint_count
0,Managing an account,70702.0
1,Incorrect information on your report,51548.0
2,Improper use of your report,40341.0
3,Problem with a purchase shown on your statement,35364.0
4,Problem with a company's investigation into an...,17641.0
5,Closing an account,17175.0
6,Getting a credit card,15170.0
7,"Other features, terms, or problems",14658.0
8,Problem with a lender or other company chargin...,13588.0
9,Attempts to collect debt not owed,11526.0


## 3. Heatmap — Issue × Product

Use **`issue_product_summary`**: rows = issue, columns = product (or pivot in BI), color = `complaint_count` or `pct_of_all_complaints`.

In [7]:
# Heatmap: Issue × Product (BI can pivot or use as long format)
# Built from marts.fct_complaints; after dbt run, marts.issue_product_summary also works.
heatmap = conn.execute("""
    WITH agg AS (
        SELECT issue, product, count(*) AS complaint_count
        FROM marts.fct_complaints
        WHERE issue IS NOT NULL AND product IS NOT NULL
        GROUP BY issue, product
    ), total AS (SELECT sum(complaint_count) AS t FROM agg)
    SELECT agg.issue, agg.product, agg.complaint_count,
        round(100.0 * agg.complaint_count / total.t, 2) AS pct_total
    FROM agg CROSS JOIN total
    ORDER BY complaint_count DESC
""").df()
heatmap.head(20)

CatalogException: Catalog Error: Table with name issue_product_summary does not exist!
Did you mean "dim_response_types"?

LINE 3:     FROM marts.issue_product_summary
                 ^

## 4. Small multiples — Time-series by issue

Use **`issue_growth_monthly`**: one line per issue, x = `month_date`, y = `complaint_count` (and optionally `pct_change_yoy`).

In [ ]:
# Small multiples: time-series by issue
timeseries = conn.execute("""
    SELECT issue, month_date, complaint_count, pct_change_mom, pct_change_yoy
    FROM marts.issue_growth_monthly
    ORDER BY issue, month_date
""").df()
timeseries.head(20)

## 5. State map

Use **`issue_state_summary`**: geography = `state`, color/size = `complaint_count`. Filter by `issue` (and optionally product via `fact_complaints` join) in BI.

In [ ]:
# State map: state-level complaint counts (filter by issue in BI)
state_map = conn.execute("""
    SELECT state, issue, complaint_count, pct_of_issue_complaints
    FROM marts.issue_state_summary
    WHERE state != 'Unknown'
    ORDER BY complaint_count DESC
""").df()
state_map.head(20)

## 6. Interactive filters (Product, State, Time)

Use **`marts.fact_complaints`** as the main detail table: it has `product`, `state`, `date_received` / `complaint_month_date`, `issue`, etc. Point BI filters to this table and use the summary marts for visuals.

In [ ]:
# Filter dimensions available in fact_complaints
conn.execute("""
    SELECT product, state, complaint_month_date, issue,
           count(*) AS complaints
    FROM marts.fact_complaints
    WHERE complaint_month_date >= '2024-01-01'
    GROUP BY product, state, complaint_month_date, issue
    ORDER BY complaints DESC
    LIMIT 10
""").df()

## 7. % Disputed per issue (root cause / theme)

Use **`issue_dispute_ratio`**: one row per issue with `pct_disputed` and `total_complaints`.

In [ ]:
dispute_ratio = conn.execute("""
    SELECT issue, total_complaints, disputed_complaints AS disputed_count, pct_disputed
    FROM marts.dim_issues
    WHERE sub_issue IS NULL
    ORDER BY total_complaints DESC
""").df()
dispute_ratio.head(15)

In [ ]:
conn.close()